# 03. Statistical analysis

**Owner:** Anibal  \
**Inputs:** `data/processed/panel_municipality_year.parquet` and
`data/processed/fire_panel_municipality_year.parquet`, both built by `make data`  \
**Outputs:** figures to `reports/figures/`, data to `data/processed/`

## Purpose

Question A: what is the relationship between demographic ageing, rural depopulation
and fire incidence in agricultural areas?

## Setup

Every chapter starts with this identical cell. Do not add ad-hoc paths below it — put new paths in `config/paths.yml`.

In [ ]:
import sys
import warnings
from pathlib import Path

# Make the src-layout package importable when the notebook kernel is not installed with -e .
project_root = Path.cwd()
while project_root != project_root.parent and not (project_root / "pyproject.toml").is_file():
    project_root = project_root.parent
sys.path.insert(0, str(project_root / "src"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm

from wildfires.io import load_fire_panel, load_municipalities, load_panel
from wildfires.viz import apply_theme

apply_theme()
warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 140)

## Load the built panels

This chapter reads the artifacts `make data` writes. It does no cleaning of its
own: every rename, type coercion and join lives in `wildfires.pipeline` and
`wildfires.merge`, is covered by tests, and is checked against a contract before
the files are emitted (`data/processed/validation_report.md`).

If a loader raises `FileNotFoundError`, run `make data`.

### The analysis panel — fire × demography, 2019–2024

One row per `(dtcc, year)`: 278 mainland municipalities × 6 years = 1,668 rows.
The window is INE's, not ours — the *Anuário Estatístico Regional* publishes
municipality demography for 2019–2024 only.

In [ ]:
panel = load_panel()
panel.shape

In [ ]:
panel.head()

### The fire panel — ICNF × EFFIS, 2001–2025

Fire history carries no demography, so it is not capped at 2019–2024. This is what
the time-series section forecasts on.

**The 2017 seam.** ICNF changed its burned-area convention in 2017 with no overlap
year, so the panel carries both, distinctly named and never silently merged:

| Columns | Years | Measures |
|---|---|---|
| `burned_ha_*` | 2017–2025 | area that burned **within** the municipality |
| `burned_ha_*_ignited` | 2001–2016 | area of fires that **ignited** in the municipality |

A single series spanning 2001–2025 therefore has to splice the two. We do that
below, explicitly and in one place, and treat the result as a splice rather than
a measurement — the level shift at 2017 is partly definitional.

In [ ]:
fire_panel = load_fire_panel()
fire_panel.shape

### Municipality names and areas

The panel keys on `dtcc`, never on name — 308 municipalities share only 306
distinct names, and GADM strips some internal spaces. Names are attached here for
labelling plots only.

Area comes from GADM geometry measured in EPSG:3763, carried on the panel as
`municipality_area_km2`. It replaces the data.gov.pt dimensions table, which is
not redistributable and so is not part of `make fetch`.

In [ ]:
municipality_names = (
    load_municipalities(mainland_only=True)[["dtcc", "municipality"]]
    .drop_duplicates("dtcc")
    .reset_index(drop=True)
)

# The 2017 convention seam, spliced in exactly one place. Both sides are the same
# unit (hectares) and only one is ever populated for a given year, so fillna is a
# concatenation across the seam, not an average of two overlapping measures.
SPLICED_AREA_COLUMNS = {
    "burned_area_ha": ("burned_ha_total", "burned_ha_total_ignited"),
    "agricultural_burned_area_ha": ("burned_ha_agric", "burned_ha_agric_ignited"),
}


def splice_burned_area(df):
    """Add the spliced 2001-2025 burned-area columns to a copy of ``df``."""
    out = df.copy()
    for target, (within, ignited) in SPLICED_AREA_COLUMNS.items():
        out[target] = out[ignited].fillna(out[within])
    return out


municipality_names.head()

## Exploratory visualisations

### Fact checking the metrics — study case: Penedono

In [ ]:
PENEDONO = municipality_names.loc[
    municipality_names["municipality"].eq("Penedono"), "dtcc"
].item()

AGE_GROUPS = {
    "pop_0_14": "0 a 14 anos",
    "pop_15_24": "15 a 24 anos",
    "pop_25_64": "25-64 anos",
    "pop_65_plus": "65 e mais anos",
    "pop_75_plus": "75 e mais anos",
}

pop_penedono = (
    panel.loc[panel["dtcc"].eq(PENEDONO), ["year", *AGE_GROUPS]]
    .rename(columns=AGE_GROUPS)
    .set_index("year")
    .sort_index()
)
pop_penedono

In [ ]:
pop_penedono

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

for column in AGE_GROUPS.values():
    ax.plot(pop_penedono.index, pop_penedono[column], marker="o", label=column)

ax.set_title("Population by age group — Penedono")
ax.set_xlabel("Year")
ax.set_ylabel("Population")
ax.legend()
ax.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

In [ ]:
fire_penedono = (
    fire_panel.loc[fire_panel["dtcc"].eq(PENEDONO), ["year", "n_fires"]]
    .set_index("year")
    .sort_index()
)
fire_penedono.tail()

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 7))

for column in AGE_GROUPS.values():
    ax1.plot(pop_penedono.index, pop_penedono[column], marker="o", linewidth=2, label=column)

ax1.set_xlabel("Year")
ax1.set_ylabel("Population")
ax1.grid(True, alpha=0.3)

# Fires span 2001-2025; demography only 2019-2024. Plotting both on one x-axis is
# deliberate — the point is that the demographic window is a small slice of the
# fire record — but do not read the pre-2019 fire line as having a demographic
# counterpart.
ax2 = ax1.twinx()
ax2.plot(
    fire_penedono.index,
    fire_penedono["n_fires"],
    color="red",
    linewidth=5,
    alpha=0.8,
    label="Number of rural fires",
)
ax2.set_ylabel("Number of rural fires", color="red")
ax2.tick_params(axis="y", labelcolor="red")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

ax1.set_title("Population structure and rural fires — Penedono")
fig.tight_layout()
plt.show()

**Conclusion:** Data shows a specific spike in 2021 both in ageing of the population and number of rural fires which may indicate the a link between the two variables or simply chance. However the number of fires was low in 2025, while in reality, the municipality of Penedono was devastated in 2025. Therefore, we conclude that the number of fires may not represent a good metric to represent the damage. Let's investigate other metrics.

### Burned area ratio - Penedono

In this chapter, we investigate the burned area ratio as a severity measure. This metric has been used in scientific research related to socio-economic impact (Chas-Amil et al., 2022 https://www.sciencedirect.com/science/article/pii/S0169204622002262), which is seems adequate to our research question.

In [ ]:
# Municipality-year burned-area ratio, 2001-2025, from the built fire panel.
burned_area_ratio = splice_burned_area(fire_panel).merge(
    municipality_names, on="dtcc", how="left", validate="many_to_one"
)
burned_area_ratio["municipality_area_ha"] = burned_area_ratio["municipality_area_km2"] * 100
burned_area_ratio["burned_area_ratio"] = (
    burned_area_ratio["burned_area_ha"] / burned_area_ratio["municipality_area_ha"]
)
burned_area_ratio["burned_area_ratio_pct"] = burned_area_ratio["burned_area_ratio"] * 100

# Ageing counts exist only for 2019-2024, so this left join leaves them missing
# for the earlier years rather than filling them.
burned_area_ratio = burned_area_ratio.merge(
    panel[["dtcc", "year", "pop_65_plus", "pop_75_plus"]].rename(
        columns={"pop_65_plus": "population_over_65", "pop_75_plus": "population_over_75"}
    ),
    on=["dtcc", "year"],
    how="left",
    validate="one_to_one",
)

burned_area_ratio = burned_area_ratio[
    [
        "dtcc",
        "municipality",
        "year",
        "burned_area_ha",
        "municipality_area_km2",
        "municipality_area_ha",
        "burned_area_ratio",
        "burned_area_ratio_pct",
        "population_over_65",
        "population_over_75",
    ]
].sort_values(["municipality", "year"]).reset_index(drop=True)

burned_area_ratio

In [ ]:
penedono_ratio = burned_area_ratio.loc[
    burned_area_ratio["dtcc"].eq(PENEDONO),
    ["year", "burned_area_ratio_pct", "population_over_65", "population_over_75"],
].sort_values("year")

fig, ax1 = plt.subplots(figsize=(12, 7))

ax1.plot(
    penedono_ratio["year"],
    penedono_ratio["burned_area_ratio_pct"],
    color="firebrick",
    marker="o",
    linewidth=2.5,
    label="Burned area ratio",
)
ax1.set_xlabel("Year")
ax1.set_ylabel("Burned area ratio (%)", color="firebrick")
ax1.tick_params(axis="y", labelcolor="firebrick")
ax1.grid(True, alpha=0.3)

ax2 = ax1.twinx()
ax2.plot(
    penedono_ratio["year"],
    penedono_ratio["population_over_65"],
    color="steelblue",
    marker="o",
    linewidth=2,
    label="Population over 65",
)
ax2.plot(
    penedono_ratio["year"],
    penedono_ratio["population_over_75"],
    color="darkorange",
    marker="o",
    linewidth=2,
    label="Population over 75",
)
ax2.set_ylabel("Older population (residents)")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

ax1.set_title("Burned area ratio and demographic ageing — Penedono")
fig.tight_layout()
plt.show()

**Conclusion:** The percentage of burned area seems to represent a better metric for the damage caused by the fire (spike in 2025 correspond to the catastrophic fires over the last year). However, we cannot conclude if the demographic ageing after 2020 have contributed to these fires in 2025. Therefore, we should verify if there is a pattern over the country.

### Burned area ratio - Country level

In [ ]:
# Country-level burned-area ratio, 2001-2025.
#
# The denominator is the mainland area the fire panel actually covers -- the sum of
# the 278 municipality areas -- not Portugal's total area. Including Madeira and the
# Azores in the denominator while ICNF covers only the mainland would deflate every
# ratio by roughly the islands' share of national area.
national_fire = splice_burned_area(fire_panel)
mainland_area_km2 = fire_panel.drop_duplicates("dtcc")["municipality_area_km2"].sum()
mainland_area_ha = mainland_area_km2 * 100

country_burned_area = (
    national_fire.groupby("year", as_index=False)["burned_area_ha"].sum(min_count=1)
)
country_burned_area["mainland_area_km2"] = mainland_area_km2
country_burned_area["mainland_area_ha"] = mainland_area_ha
country_burned_area["burned_area_ratio"] = (
    country_burned_area["burned_area_ha"] / country_burned_area["mainland_area_ha"]
)
country_burned_area["burned_area_ratio_pct"] = country_burned_area["burned_area_ratio"] * 100

country_older_population = (
    panel.groupby("year", as_index=False)["pop_65_plus"]
    .sum(min_count=1)
    .rename(columns={"pop_65_plus": "population_over_65"})
)

portugal_burned_area_ratio = (
    country_burned_area
    .merge(country_older_population, on="year", how="left", validate="one_to_one")
    [[
        "year",
        "burned_area_ha",
        "mainland_area_km2",
        "mainland_area_ha",
        "burned_area_ratio",
        "burned_area_ratio_pct",
        "population_over_65",
    ]]
    .sort_values("year")
    .reset_index(drop=True)
)

portugal_burned_area_ratio

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 7))

ax1.plot(
    portugal_burned_area_ratio["year"],
    portugal_burned_area_ratio["burned_area_ratio_pct"],
    color="firebrick",
    marker="o",
    linewidth=2.5,
    label="Burned area ratio",
)
ax1.set_xlabel("Year")
ax1.set_ylabel("Burned area ratio (%)", color="firebrick")
ax1.tick_params(axis="y", labelcolor="firebrick")
ax1.grid(True, alpha=0.3)

ax2 = ax1.twinx()
ax2.plot(
    portugal_burned_area_ratio["year"],
    portugal_burned_area_ratio["population_over_65"],
    color="steelblue",
    marker="o",
    linewidth=2,
    label="Population over 65",
)
ax2.set_ylabel("Population over 65 (residents)", color="steelblue")
ax2.tick_params(axis="y", labelcolor="steelblue")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

ax1.set_title("Portugal Burned Area Ratio and Population Ageing")
fig.tight_layout()
plt.show()

**Conclusion:** Over the country we can observe ciclic patterns of fire damage incidence. The population over 65 is drastically increasing in last years. However, we do not have data about population demographics before 2019, which makes it hard to draw conclusions about the relationship between the two variables. We therefore proceed with additional analysis.

## Analysis panel

The panel below selects the fire and demographic variables this question needs. It
selects and transforms only — the aggregation to municipality-year, the `dtcc`
normalisation and the three-source join all happened upstream in `make data`, so
this chapter cannot silently disagree with chapters 02 and 04 about what a row means.

In [ ]:
# Panel for the fixed-effects model: demographic ageing, rural depopulation and
# agricultural fire outcomes. Everything comes off the built panel; the only
# derivation here is the modelling transform.
analysis_panel = splice_burned_area(panel).merge(
    municipality_names, on="dtcc", how="left", validate="many_to_one"
)
analysis_panel = analysis_panel.rename(
    columns={
        "n_fires": "rural_fire_count",
        "pop_total": "population_total",
        "pop_65_plus": "population_over_65",
        "pop_75_plus": "population_over_75",
    }
)[
    [
        "dtcc",
        "municipality",
        "year",
        "agricultural_burned_area_ha",
        "rural_fire_count",
        "municipality_area_km2",
        "population_total",
        "population_over_65",
        "population_over_75",
    ]
]

analysis_panel

Now we define population growth as a proxy to measure rural depopulation: Population growth = Population_t - Population_t-1 / Population_t-1

This allows to capture demographic change rather than population size.

In [ ]:
analysis_panel["municipality_area_ha"] = analysis_panel["municipality_area_km2"] * 100
analysis_panel["agri_burned_ratio_pct"] = (
    analysis_panel["agricultural_burned_area_ha"]
    / analysis_panel["municipality_area_ha"]
    * 100
)
analysis_panel["share_over_65"] = (
    analysis_panel["population_over_65"] / analysis_panel["population_total"]
)
analysis_panel["share_over_75"] = (
    analysis_panel["population_over_75"] / analysis_panel["population_total"]
)
analysis_panel = analysis_panel.sort_values(["dtcc", "year"]).reset_index(drop=True)

# pct_change within municipality. 2019 is NaN for every municipality by
# construction: it is the first year in the window, so it has no predecessor.
analysis_panel["population_growth"] = analysis_panel.groupby("dtcc")[
    "population_total"
].pct_change()
analysis_panel["log_fire_count"] = np.log1p(
    analysis_panel["rural_fire_count"].clip(lower=0)
)

analysis_panel = analysis_panel.replace([np.inf, -np.inf], np.nan)
analysis_panel

In [ ]:
plt.figure(figsize=(8,5))
plt.hist(
    analysis_panel["agri_burned_ratio_pct"].dropna(),
    bins=50,
    edgecolor="black"
)
plt.xlabel("Agricultural burned area ratio (%)")
plt.ylabel("Frequency")
plt.title("Distribution of agricultural burned area ratio")
plt.show()

As expected, it is extremely skewed. We need to do log transformation before proceeding.

In [ ]:
# Use log1p because agricultural burned-area ratios are strongly right-skewed.
model_data = analysis_panel.copy()
model_data["log_agri_burned_ratio"] = np.log1p(
    model_data["agri_burned_ratio_pct"].clip(lower=0)
)

Now we can define municipality and year as fixed effects because of topography, climate, etc. Without these controls, differences between municipalities could bias the ageing coefficient.

In [ ]:
def fit_fixed_effects(data, predictors, outcome="log_agri_burned_ratio"):
    required = ["dtcc", "year", outcome, *predictors]
    sample = data[required].dropna().copy()
    design = sample[predictors].copy()
    design = pd.concat(
        [
            design,
            pd.get_dummies(sample["dtcc"], prefix="municipality", drop_first=True, dtype=float),
            pd.get_dummies(sample["year"], prefix="year", drop_first=True, dtype=float),
        ],
        axis=1,
    )
    design = sm.add_constant(design.astype(float), has_constant="add")
    fit = sm.OLS(sample[outcome].astype(float), design).fit(
        cov_type="cluster",
        cov_kwds={"groups": sample["dtcc"]},
    )
    return fit, sample, design

ageing_model, ageing_sample, ageing_design = fit_fixed_effects(
    model_data,
    ["share_over_65"],
)
ageing_growth_model, ageing_growth_sample, ageing_growth_design = fit_fixed_effects(
    model_data,
    ["share_over_65", "population_growth"],
)

model_comparison = pd.DataFrame(
    {
        "model": ["Ageing + municipality/year effects", "Ageing + population growth + effects"],
        "observations": [len(ageing_sample), len(ageing_growth_sample)],
        "r_squared": [ageing_model.rsquared, ageing_growth_model.rsquared],
        "share_over_65_coef": [
            ageing_model.params["share_over_65"],
            ageing_growth_model.params["share_over_65"],
        ],
        "share_over_65_pvalue": [
            ageing_model.pvalues["share_over_65"],
            ageing_growth_model.pvalues["share_over_65"],
        ],
        "population_growth_coef": [
            np.nan,
            ageing_growth_model.params["population_growth"],
        ],
        "population_growth_pvalue": [
            np.nan,
            ageing_growth_model.pvalues["population_growth"],
        ],
    }
)
model_comparison

The coefficient on `share_over_65` is 0.661 and statistically significant at the 5%
level (p = 0.042). A municipality whose share of residents over 65 rises by one
percentage point is associated with roughly a 0.7% increase in agricultural
burned-area intensity. Adding population growth, the ageing coefficient stays
positive (0.670) but its significance weakens (p = 0.080), and R² rises only
modestly, from 0.204 to 0.230.

Note the sample sizes. The first model fits 1,666 of the panel's 1,668 rows — two
municipality-years have no agricultural burned area recorded. The second drops to
1,388 because `population_growth` is undefined for 2019, the first year in the
window: 278 municipalities × 5 usable years. That is a structural consequence of
INE's six-year publication window, not attrition we could recover.

**Conclusion:** the results provide limited evidence that demographic structure is
associated with agricultural fire outcomes. Both the magnitude and the statistical
significance of the effects are modest, and the second specification's weakened
p-value warns against reading the first one too confidently.

The demographic panel is limited to 2019–2024. Next we forecast Portugal's annual
burned-area ratio using only its own history.

## Fire-only time-series forecasting

The 2001-2022 period is used for model development and 2023-2025 is held out for evaluation.

### Define and validate the target

In [ ]:
country_series = (
    portugal_burned_area_ratio.set_index("year")["burned_area_ratio_pct"]
    .sort_index()
    .astype(float)
)
expected_years = pd.Index(range(2001, 2026), name="year")
assert country_series.index.equals(expected_years)
assert country_series.notna().all()

country_series.to_frame("burned_area_ratio_pct").head()

### Chronological train-test split

The model is trained on 2001-2022 and evaluated on the untouched 2023-2025 period. This prevents later observations from influencing model selection and tests whether past fire ratios could anticipate recent years.

In [ ]:
train = country_series.loc[:2022]
test = country_series.loc[2023:2025]

split_summary = pd.DataFrame(
    {
        "period": ["training", "test"],
        "first_year": [train.index.min(), test.index.min()],
        "last_year": [train.index.max(), test.index.max()],
        "observations": [len(train), len(test)],
    }
)
split_summary

### Inspect autocorrelation and choose differencing

The raw series is non-negative and contains large spikes. We inspect the raw series and its first difference. The ACF/PACF are used as low-order ARIMA guides, not as an automatic high-order parameter search because annual data provides only 22 training observations.

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller

raw_adf = adfuller(train, autolag="AIC")
differenced_train = train.diff().dropna()
diff_adf = adfuller(differenced_train, autolag="AIC")

stationarity_checks = pd.DataFrame(
    {
        "series": ["level", "first_difference"],
        "ADF_statistic": [raw_adf[0], diff_adf[0]],
        "ADF_pvalue": [raw_adf[1], diff_adf[1]],
    }
)

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
axes[0, 0].plot(train.index, train, marker="o", color="firebrick")
axes[0, 0].set_title("Training series")
axes[0, 0].set_ylabel("Burned area ratio (%)")
axes[0, 1].plot(differenced_train.index, differenced_train, marker="o", color="darkgreen")
axes[0, 1].axhline(0, color="black", linewidth=0.8)
axes[0, 1].set_title("First difference")
axes[0, 1].set_ylabel("Change in ratio (percentage points)")
plot_acf(train, lags=8, ax=axes[1, 0], zero=False)
axes[1, 0].set_title("ACF: level series")
plot_acf(differenced_train, lags=8, ax=axes[1, 1], zero=False)
axes[1, 1].set_title("ACF: first difference")
fig.tight_layout()
plt.show()

stationarity_checks

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
plot_pacf(train, lags=8, ax=axes[0], method="ywm", zero=False)
axes[0].set_title("PACF: level series")
plot_pacf(differenced_train, lags=8, ax=axes[1], method="ywm", zero=False)
axes[1].set_title("PACF: first difference")
fig.tight_layout()
plt.show()

The level ACF and PACF show no strong low-order spike, so begin with a mean-only model and low-order ARMA alternatives. The differenced plots are retained as a sensitivity check, but they do not justify forcing d=1. Because there are only 22 training observations, model selection will use rolling one-step errors rather than a large ARIMA grid.

In [ ]:
arima_orders = {
    "ARIMA(0,0,0)": (0, 0, 0),
    "ARIMA(1,0,0)": (1, 0, 0),
    "ARIMA(0,0,1)": (0, 0, 1),
    "ARIMA(1,0,1)": (1, 0, 1),
}


def arima_forecast(series, order, steps=1, alpha=0.05):
    fitted = sm.tsa.ARIMA(series, order=order, trend="c").fit()
    result = fitted.get_forecast(steps=steps)
    mean = np.asarray(result.predicted_mean, dtype=float)
    interval = result.conf_int(alpha=alpha)
    lower = interval.iloc[:, 0].to_numpy(dtype=float)
    upper = interval.iloc[:, 1].to_numpy(dtype=float)
    return mean, lower, upper


def one_step_forecast(series, model_name):
    if model_name == "historical_mean":
        return float(series.mean())
    if model_name == "naive":
        return float(series.iloc[-1])
    return float(arima_forecast(series, arima_orders[model_name])[0][0])

candidate_models = ["historical_mean", "naive", *arima_orders]
candidate_models

### Select the model with rolling one-step validation

Each candidate is repeatedly fitted using data available up to year t and predicts year t+1. This imitates real forecasting and prevents choosing a model because it performs well only on the final holdout.

Let's use the rolling-origin time-series validation (or walk-forward validation). Basically, instead of fitting the model once, it reapeatedly use first observations to training history and predicts next observation. The expands the training series by one observation until the end.

In [ ]:
rolling_rows = []
for model_name in candidate_models:
    actual = []
    predicted = []
    for end_position in range(10, len(train)):
        history = train.iloc[:end_position]
        try:
            prediction = one_step_forecast(history, model_name)
        except (ValueError, np.linalg.LinAlgError):
            prediction = np.nan
        actual.append(float(train.iloc[end_position]))
        predicted.append(prediction)

    rolling_frame = pd.DataFrame({"actual": actual, "predicted": predicted}).dropna()
    rolling_rows.append(
        {
            "model": model_name,
            "validation_observations": len(rolling_frame),
            "MAE": (rolling_frame["predicted"] - rolling_frame["actual"]).abs().mean(),
            "RMSE": np.sqrt(
                ((rolling_frame["predicted"] - rolling_frame["actual"]) ** 2).mean()
            ),
        }
    )

rolling_scores = (
    pd.DataFrame(rolling_rows)
    .sort_values("RMSE")
    .reset_index(drop=True)
)
selected_model = rolling_scores.iloc[0]["model"]
rolling_scores

**Conclusion:** The best predictor is the long-run mean of the series. The simplest model performed best because data do not exhibit a strong predictable temporal pattern. Knowing previous years does not substantially improve prediction relative to simply using the long-run average.

Let's what happens visually.

### Forecast the 2023-2025 years

The selected model is refitted using all training observations through 2022. The final comparison keeps 2023-2025 untouched until this point. Prediction intervals show the range of values compatible with the model, not a guarantee that extreme fires will be covered.

In [ ]:
# forecast 2024-2025 using fire history
forecast_train = country_series.loc[:2023]
forecast_years = pd.Index([2024, 2025], name="year")

if selected_model == "historical_mean":
    forecast_values = np.repeat(forecast_train.mean(), len(forecast_years))
elif selected_model == "naive":
    forecast_values = np.repeat(forecast_train.iloc[-1], len(forecast_years))
else:
    forecast_values = arima_forecast(
        forecast_train,
        arima_orders[selected_model],
        steps=len(forecast_years),
    )[0]

forecast_2024_2025 = pd.DataFrame(
    {
        "actual": country_series.loc[forecast_years].to_numpy(),
        "predicted": forecast_values,
    },
    index=forecast_years,
)
forecast_2024_2025["absolute_error"] = (
    forecast_2024_2025["actual"] - forecast_2024_2025["predicted"]
).abs()

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(
    country_series.loc[:2023].index,
    country_series.loc[:2023],
    color="black",
    marker="o",
    label="Actual burned-area ratio",
)
ax.plot(
    forecast_2024_2025.index,
    forecast_2024_2025["actual"],
    color="firebrick",
    marker="o",
    linewidth=2.5,
    label="Actual 2024-2025",
)
ax.plot(
    [2023, *forecast_2024_2025.index],
    [country_series.loc[2023], *forecast_2024_2025["predicted"]],
    color="steelblue",
    marker="x",
    linestyle="--",
    linewidth=2,
    label=f"Predicted 2024-2025: {selected_model}",
)
ax.axvline(2023.5, color="grey", linestyle=":", label="Forecast origin")
ax.set_xlabel("Year")
ax.set_ylabel("Burned area ratio (%)")
ax.set_title("Actual versus Predicted Portugal Burned-Area Ratio")
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
plt.show()

forecast_2024_2025

**Conclusion:** The 2025 increase was not predictable from the historical pattern contained in the series. The simple model assumed 2025 would be an "average" year. Instead, 2025 was much higher than average.

## Overall conclusion

The analysis explored whether population ageing and population decline are linked to agricultural fire outcomes.

Results suggest a possible relationship with burned area, but the evidence is not strong enough to draw firm conclusions. Municipalities with a higher share of older residents tended to have slightly higher agricultural burned areas, but the effect became weaker when population change was included in the model.

Importantly, the results were different for fire occurrence and fire impact:
* We found no clear link between demographic change and the number of fires.
* We found some indication that demographic change may be related to how much land burns when fires occur.

At the national level, past fire patterns were not very useful for forecasting future burned areas. A simple average-based forecast performed as well as more complex models, and it did not anticipate the large increase observed in 2025.

Overall, the current data does not provide strong evidence that ageing or depopulation directly increase fire risk, but it also does not rule out a relationship.

### Why the link may not be clear

Several factors may explain the inconclusive results:
* The demographic dataset covers only a short period (2019-2024).
* Effects of ageing and depopulation may take many years to appear.
* Key factors such as land abandonment, vegetation growth, weather conditions, and firefighting capacity were not directly measured.
* Fire outcomes are heavily influenced by drought, wind, and other environmental conditions that can outweigh demographic effects.
* Neighbor municipalities were not considered (e.g. a fire may pass from one municipality to the other but this relationship was not modeled).

### Recommendations for future analysis

A stronger assessment would require:
* Longer demographic and population histories.
* Better measures of rural decline and land abandonment.
* Weather and drought information.
* Land management and vegetation indicators.
* Testing delayed effects, where demographic change influences fire outcomes several years later.

With these additional data sources, it would be possible to determine more confidently whether demographic change contributes to agricultural fire risk or severity.